# Ollama con GPU en Colab, para tu dashboard local

Deja Ollama corriendo aqui, con GPU, y publica una URL publica con `ngrok` para que tu dashboard (en tu portatil) le hable a este Ollama en vez de al tuyo local, que va por CPU.

**Antes de nada:** Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> Acelerador por hardware -> **GPU (T4)**.

**Necesitas una cuenta gratuita de ngrok** (https://ngrok.com -> Sign up), y tu authtoken personal, en Dashboard -> Your Authtoken. Es gratis, un minuto.

## Paso 1: instalar y arrancar Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
import shutil
import subprocess
import time

# Instala Ollama si la celda de instalación no se ejecutó o quedó fuera del PATH.
if shutil.which("ollama") is None:
    subprocess.run(
        ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
        check=True,
    )
    os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")

ollama_bin = shutil.which("ollama")
if ollama_bin is None:
    raise FileNotFoundError(
        "No se encontró Ollama después de instalarlo. Ejecuta de nuevo esta celda."
    )

# OLLAMA_ORIGINS="*" permite que Ollama acepte peticiones a través de ngrok.
entorno = os.environ.copy()
entorno["OLLAMA_ORIGINS"] = "*"

proceso_ollama = subprocess.Popen(
    [ollama_bin, "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
    env=entorno,
)
time.sleep(5)
print("Ollama arrancado en segundo plano (PID", proceso_ollama.pid, ")")
print("Ejecutable:", ollama_bin)

In [ ]:
!ollama pull llama3.2:3b

# Opcional: con GPU de verdad, qwen3:8b puede merecer la pena otra vez
# (en tu CPU local salio peor que llama3.2:3b: mas parametros, mas lento sin GPU.
# Con GPU la relacion puede invertirse -- se puede probar sin miedo.)
# !ollama pull qwen3:8b

In [ ]:
# Confirma que Ollama vera la GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Paso 2: publicar Ollama con ngrok
Pega tu authtoken de ngrok en la siguiente celda antes de ejecutarla.

In [ ]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok

NGROK_AUTHTOKEN = "PEGA_AQUI_TU_AUTHTOKEN"  # de https://dashboard.ngrok.com/get-started/your-authtoken

ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunel = ngrok.connect(11434)
print('URL publica de Ollama:', tunel.public_url)
print()
print('Copia esta URL en el campo "URL de Ollama remoto" de la barra lateral de tu dashboard.')
print('Prueba de salud:', __import__('requests').get(tunel.public_url + '/api/tags', timeout=30).status_code)

## Paso 3: mantener esta sesión viva

Mientras quieras usar el dashboard local con esta GPU, **deja esta pestaña de Colab abierta** y no dejes que se quede inactiva demasiado tiempo (Colab gratuito desconecta sesiones inactivas tras ~90 minutos). La celda siguiente hace una comprobación periódica -- para de ejecutarla (botón de stop) cuando termines de usar el dashboard.

In [ ]:
import time

print('Ollama activo en:', tunel.public_url)
print('Pulsa el boton de stop de esta celda cuando termines de usar el dashboard.')
print()

try:
    while True:
        time.sleep(60)
        print('.', end='', flush=True)
except KeyboardInterrupt:
    print('\nDetenido.')

## Si la URL cambia

Cada vez que reinicies esta sesión de Colab, `ngrok.connect()` te da una URL **nueva** (el plan gratuito no permite una URL fija). Si reinicias, vuelve a ejecutar desde el Paso 2 y actualiza la URL en el dashboard.